In [2]:
from langchain_groq import ChatGroq 
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnableLambda,
    RunnableBranch,
    RunnableParallel,
    RunnableMap,
    RunnableSequence
)

In [3]:
# Install Groq SDK if not already installed
# pip install groq langchain
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

# Initialize Groq LLM
llm = ChatGroq(
    temperature=0.7,
    groq_api_key=groq_api_key,  # Replace with your actual key
    model_name="openai/gpt-oss-120b"          # Example Groq model
)

**Basic Langchain**

In [ ]:
# Example usage
response = llm.invoke("Explain hybrid retrieval in RAG systems in 5 numbered points.")
print(response.content)


**Hybrid Retrieval in Retrieval‑Augmented Generation (RAG) – 5 Key Points**

1. **Dual‑Indexer Architecture**  
   - **Dense (vector) index**: embeddings generated by a neural encoder (e.g., BERT, SBERT, DPR) capture semantic similarity.  
   - **Sparse (keyword) index**: traditional inverted‑file structures (BM25, TF‑IDF, Elasticsearch) capture exact term matches and lexical overlap.  
   - The system maintains both indices in parallel, allowing each query to be processed by both retrieval pathways.

2. **Parallel Query Execution**  
   - When a user query arrives, it is **simultaneously** sent to the dense encoder (producing a query embedding) and to the sparse engine (producing a term‑based query).  
   - Each engine returns its own top‑k candidate documents (e.g., 50 from dense, 50 from sparse). This parallelism preserves the speed of sparse retrieval while leveraging the semantic breadth of dense retrieval.

3. **Fusion & Re‑ranking**  
   - The two candidate sets are **merged** (

In [ ]:
template = "Give me 3 career skills that are in high demand in {year}."
prompt = PromptTemplate.from_template(template)

chain = prompt | llm | StrOutputParser()

response = chain.invoke({"year": "2026"})
print(response)

**Three career skills that are in especially high demand in 2026**

| # | Skill | Why it’s hot in 2026 | Typical roles that need it |
|---|-------|----------------------|----------------------------|
| 1 | **Generative AI & Prompt Engineering** | • Most companies now embed large‑language‑model (LLM) or multimodal AI tools into products, services, and internal workflows. <br>• The market has shifted from “building models” to “designing, fine‑tuning, and safely deploying prompt‑driven solutions.” <br>• Talent that can translate business problems into effective prompts, evaluate model outputs, and mitigate bias/ethics risks is scarce. | AI Product Manager, Prompt Engineer, LLM‑Ops Engineer, AI‑Enabled UX Designer, Business Analyst (AI‑focused) |
| 2 | **Cyber‑Resilience Architecture** | • Ransomware, supply‑chain attacks, and deep‑fake‑enabled social engineering have grown in frequency and sophistication. <br>• Organizations are moving from reactive “cybersecurity” to proactive “cyber‑res

In [14]:
template = "Give me 3 most common phrases used in {country}."
prompt = PromptTemplate.from_template(template)

chain = prompt | llm | StrOutputParser()

# response = chain.invoke({"country": "the United States"})
# response = chain.invoke({"country": "India"})
response = chain.invoke({"country": "France"})

print(response)

Here are three everyday French expressions that you’ll hear all over France, along with their literal translations and typical English equivalents:

| French phrase | Literal translation | Typical English equivalent | When to use it |
|--------------|---------------------|----------------------------|----------------|
| **« Bonjour »** | “Good day” | “Hello / Good morning” | As a standard greeting when you meet someone (any time before the evening). |
| **« Merci ! »** | “Thanks!” | “Thank you!” | To express gratitude, whether for a small favor or a larger help. |
| **« Ça va ?** » | “It goes?” | “How are you? / How’s it going?” | A casual way to ask someone how they’re feeling; the usual reply is “Ça va” (I’m fine). |

These three are the backbone of polite, everyday conversation in French‑speaking settings. Feel free to sprinkle them into your interactions—they’re instantly recognizable and always appreciated!


**Pipelines Runnables**

In [16]:
uppercase = RunnableLambda(lambda x: x.upper())
add_exclaim = RunnableLambda(lambda x: x + "!")

pipeline = uppercase | add_exclaim

pipeline.invoke("hello")

'HELLO!'

In [28]:
# parallel = RunnableMap({
#     "upper": RunnableLambda(lambda x: x.upper()),
#     "lower": RunnableLambda(lambda x: x.lower())
# })

parallel = RunnableParallel({
    "upper": RunnableLambda(lambda x: x.upper()),
    "lower": RunnableLambda(lambda x: x.lower())
})

parallel.invoke("Hello")

{'upper': 'HELLO', 'lower': 'hello'}

In [22]:
prompt = ChatPromptTemplate.from_template(
    "Translate this to Hindi: {text}"
)

pipeline = prompt | llm | StrOutputParser()

pipeline.invoke({"text": "Hello, how are you?"})

'**हिंदी में अनुवाद:**  \n"Hello, how are you?" → **"नमस्ते, आप कैसे हैं?"**'

**Conditionals**

In [22]:
topic = "Kolkata"          # ["math", "algebra", "geometry", "calculus"]
subtopic = "Parallelograms"

In [23]:

# Step 1: Define LLM

# Step 2: Define prompt templates

math_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    You are an expert mathematics tutor.

    Follow this structure:
    1. Concept explanation
    2. Mathematical formula
    3. Step-by-step numerical example
    4. Final answer
    5. One practice question on calculus

    Always show calculations and explain each step.
    """),
    ("human", "Explain {topic} \n Subtopic to explain: {subtopic}")
])

general_prompt = PromptTemplate(
    input_variables=["topic"],
    template="You are a general knowledge tutor. Explain the topic {topic} in 6 points."
)

# Step 3: Define conditional functions

def is_math(inputs):
    topic = inputs["topic"].lower()

    return any(
        word in topic
        for word in ["math", "algebra", "geometry", "calculus"]
    )

# Step 4: RunnableBranch

branch = RunnableBranch(
    (
        is_math,
        math_prompt | llm
    ),
    general_prompt | llm  # Default branch
)

# Step 5: RunnableParallel

parallel_tasks = RunnableParallel({
    "short_answer": RunnableLambda(
        lambda x: f"Topic requested: {x['topic']}"
    ),
    "detailed_explanation": branch
})

# Step 6: Merge results

merge_results = RunnableLambda(
    lambda inputs: (
        f"{inputs['short_answer']}\n\n"
        f"Detailed explanation:\n"
        f"{inputs['detailed_explanation'].content}"
    )
)

# Step 7: Full chain

full_chain = parallel_tasks | merge_results

# Step 8: Execute

result = full_chain.invoke({"topic": topic, "subtopic": subtopic})
print(result)


Topic requested: Kolkata

Detailed explanation:
**Kolkata (formerly Calcutta) – Six Key Points**

1. **Historical Capital of British India**  
   - Founded in 1690 as a trading post of the British East India Company.  
   - Became the capital of British India in 1772 (officially 1773) and remained so until 1911, shaping its architecture, institutions, and cosmopolitan character.

2. **Cultural Hub – “City of Joy” & “Cultural Capital of India”**  
   - Renowned for literature (Rabindranath Tagore, Satyajit Ray), music, theatre, and the annual **Durga Puja** festival, which draws millions of visitors.  
   - Home to iconic art venues like the **Academy of Fine Arts**, **Nandan** film complex, and the **Kalighat** temple.

3. **Educational & Intellectual Center**  
   - Hosts some of India’s oldest and most prestigious institutions: **University of Calcutta** (1857), **Jadavpur University**, **Indian Institute of Management Calcutta (IIM‑C)**, and the **Indian Statistical Institute**.  
 

Why this works:
1. is_math returns a Boolean.
2. If True, math_prompt | llm executes.
3. If False, general_prompt | llm executes as the default.
4. Both outputs from RunnableParallel are combined into a dictionary.
5. merge_results combines the dictionary into a final string.